In [33]:

import pandas as pd

print("Pandas imported successfully!")


Pandas imported successfully!


In [34]:

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:

import zipfile

zip_path = "/content/drive/MyDrive/train_ver2.csv.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/")

print("Extraction completed!")


Extraction completed!


In [19]:

import pandas as pd

chunks = pd.read_csv(
    "/content/train_ver2.csv",
    chunksize=100000,
    low_memory=False
)

print("Dataset ready!")


Dataset ready!


In [20]:

last_month = []

for chunk in chunks:
    filtered = chunk[chunk["fecha_dato"] == "2016-05-28"]
    last_month.append(filtered)

df_last = pd.concat(last_month, ignore_index=True)

print(df_last.shape)
df_last.head()


(931453, 48)


,fecha_dato,ncodpers,ind_empleado,pais_residencia,sexo,age,fecha_alta,ind_nuevo,antiguedad,indrel,...,ind_hip_fin_ult1,ind_plan_fin_ult1,ind_pres_fin_ult1,ind_reca_fin_ult1,ind_tjcr_fin_ult1,ind_valo_fin_ult1,ind_viv_fin_ult1,ind_nomina_ult1,ind_nom_pens_ult1,ind_recibo_ult1
0,2016-05-28,657640,N,ES,H,68,2006-11-17,0.0,114,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
1,2016-05-28,657788,N,ES,H,42,2006-11-18,0.0,114,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
2,2016-05-28,657795,N,ES,V,44,2006-11-18,0.0,114,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0
3,2016-05-28,657790,N,ES,V,42,2006-11-18,0.0,114,1.0,...,0,0,0,0,1,0,0,1.0,1.0,1
4,2016-05-28,657794,N,ES,H,49,2006-11-18,0.0,114,1.0,...,0,0,0,0,0,0,0,0.0,0.0,0


In [21]:

df_last["age"] = pd.to_numeric(
    df_last["age"], errors="coerce"
)

df_last["renta"] = pd.to_numeric(
    df_last["renta"], errors="coerce"
)

print(df_last[["age", "renta"]].head())


   age      renta
0   68  147347.97
1   42  132559.35
2   44   81399.57
3   42        NaN
4   49  102189.00


In [22]:

city_stats = df_last.groupby("nomprov").agg(
    customer_count=("ncodpers", "count"),
    average_income=("renta", "mean"),
    median_income=("renta", "median"),
    average_age=("age", "mean")
)

print(city_stats.head())


          customer_count  average_income  median_income  average_age
nomprov                                                             
ALAVA               2597   132838.933333     103321.140    46.090104
ALBACETE            7792    82970.140154      78566.325    32.976771
ALICANTE           22210    87142.959076      67332.870    38.883116
ALMERIA             4317    84851.186250      72932.925    40.165393
ASTURIAS           18332   101311.341324      87085.140    38.950524


In [23]:

# Keep cities with at least 50 customers
filtered_cities = city_stats[
    city_stats["customer_count"] >= 50
]

# Sort by average income (highest first)
top5_cities = filtered_cities.sort_values(
    by="average_income",
    ascending=False
).head(5)

# Display results
top5_cities


,customer_count,average_income,median_income,average_age
nomprov,,,,
CEUTA,513,200515.885152,128699.580,44.007797
MADRID,298841,178362.993637,138024.135,45.205564
"BALEARS, ILLES",9143,171890.951564,122837.220,39.748770
BARCELONA,88754,164584.843157,130877.160,39.865144
MELILLA,641,149071.944149,117718.365,45.329173


In [24]:

import pandas as pd

# Read only the necessary columns
columns = pd.read_csv(
    "/content/train_ver2.csv",
    nrows=0
).columns.tolist()

product_cols = columns[24:48]

use_cols = ["fecha_dato", "ncodpers"] + product_cols

chunks_q2 = pd.read_csv(
    "/content/train_ver2.csv",
    usecols=use_cols,
    chunksize=100000,
    low_memory=False
)

print("Q2 dataset reader ready!")
print("Number of products:", len(product_cols))
print(product_cols)


Q2 dataset reader ready!
Number of products: 24
['ind_ahor_fin_ult1', 'ind_aval_fin_ult1', 'ind_cco_fin_ult1', 'ind_cder_fin_ult1', 'ind_cno_fin_ult1', 'ind_ctju_fin_ult1', 'ind_ctma_fin_ult1', 'ind_ctop_fin_ult1', 'ind_ctpp_fin_ult1', 'ind_deco_fin_ult1', 'ind_deme_fin_ult1', 'ind_dela_fin_ult1', 'ind_ecue_fin_ult1', 'ind_fond_fin_ult1', 'ind_hip_fin_ult1', 'ind_plan_fin_ult1', 'ind_pres_fin_ult1', 'ind_reca_fin_ult1', 'ind_tjcr_fin_ult1', 'ind_valo_fin_ult1', 'ind_viv_fin_ult1', 'ind_nomina_ult1', 'ind_nom_pens_ult1', 'ind_recibo_ult1']


In [25]:

import sqlite3
import pandas as pd

# Create a database on disk to avoid RAM crashes
conn = sqlite3.connect("/content/q2_data.db")

# Save chunks into SQLite
for i, chunk in enumerate(chunks_q2):
    chunk[product_cols] = chunk[product_cols].apply(
        pd.to_numeric, errors="coerce"
    ).fillna(0).astype("int8")

    chunk.to_sql(
        "customer_products",
        conn,
        if_exists="replace" if i == 0 else "append",
        index=False
    )

    if i % 10 == 0:
        print("Processed chunks:", i + 1)

print("Data preparation completed!")


Processed chunks: 1
Processed chunks: 11
Processed chunks: 21
Processed chunks: 31
Processed chunks: 41
Processed chunks: 51
Processed chunks: 61
Processed chunks: 71
Processed chunks: 81
Processed chunks: 91
Processed chunks: 101
Processed chunks: 111
Processed chunks: 121
Processed chunks: 131
Data preparation completed!


In [26]:

# Create an index to speed up processing
conn.execute("""
CREATE INDEX IF NOT EXISTS idx_customer_date
ON customer_products (ncodpers, fecha_dato)
""")

conn.commit()

# Check customer records
pd.read_sql_query("""
SELECT ncodpers, fecha_dato
FROM customer_products
ORDER BY ncodpers, fecha_dato
LIMIT 10
""", conn)


,ncodpers,fecha_dato
0,15889,2015-01-28
1,15889,2015-02-28
2,15889,2015-03-28
3,15889,2015-04-28
4,15889,2015-05-28
5,15889,2015-06-28
6,15889,2015-07-28
7,15889,2015-08-28
8,15889,2015-09-28
9,15889,2015-10-28


In [27]:

# Calculate new subscriptions for each product
new_counts = {}

for product in product_cols:
    query = f"""
    WITH previous_data AS (
        SELECT
            ncodpers,
            fecha_dato,
            "{product}" AS current_value,
            LAG("{product}") OVER (
                PARTITION BY ncodpers
                ORDER BY fecha_dato
            ) AS previous_value,
            LAG(fecha_dato) OVER (
                PARTITION BY ncodpers
                ORDER BY fecha_dato
            ) AS previous_date
        FROM customer_products
    )
    SELECT fecha_dato, COUNT(*) AS new_count
    FROM previous_data
    WHERE previous_value = 0
      AND current_value = 1
      AND (
          (CAST(substr(fecha_dato, 1, 4) AS INTEGER) -
           CAST(substr(previous_date, 1, 4) AS INTEGER)) * 12
          + CAST(substr(fecha_dato, 6, 2) AS INTEGER)
          - CAST(substr(previous_date, 6, 2) AS INTEGER)
      ) = 1
    GROUP BY fecha_dato
    """

    new_counts[product] = pd.read_sql_query(query, conn)

print("New subscription counting completed!")


New subscription counting completed!


In [28]:

# Calculate total new subscriptions for each product
total_counts = {}

for product, monthly_data in new_counts.items():
    total_counts[product] = monthly_data["new_count"].sum()

# Convert results to a DataFrame
q2_results = pd.DataFrame(
    list(total_counts.items()),
    columns=["Product", "New_Subscriptions"]
)

# Sort and display the top 5 products
top5_products = q2_results.sort_values(
    by="New_Subscriptions",
    ascending=False
).head(5)

top5_products


,Product,New_Subscriptions
23,ind_recibo_ult1,153146
22,ind_nom_pens_ult1,84709
21,ind_nomina_ult1,73753
18,ind_tjcr_fin_ult1,69118
2,ind_cco_fin_ult1,69103


In [29]:

# Select the 24 banking product columns
product_cols = df_last.columns[24:48]

# Convert product values to numbers
products = df_last[product_cols].apply(
    pd.to_numeric, errors="coerce"
).fillna(0)

# Count products owned by each customer
df_last["product_count"] = products.sum(axis=1)

# 1 = owns at least 3 products, 0 = fewer than 3
df_last["has_3plus"] = (
    df_last["product_count"] >= 3
).astype(int)

# Display results
df_last[["ncodpers", "product_count", "has_3plus"]].head()


,ncodpers,product_count,has_3plus
0,657640,0.0,0
1,657788,1.0,0
2,657795,1.0,0
3,657790,6.0,1
4,657794,2.0,0


In [30]:

import numpy as np

# Convert age to numeric
df_last["age"] = pd.to_numeric(
    df_last["age"], errors="coerce"
)

# Create age groups
df_last["age_group"] = np.select(
    [
        df_last["age"] >= 65,
        df_last["age"] >= 30,
        df_last["age"] < 30
    ],
    [
        "senior",
        "adult",
        "young"
    ],
    default="unknown"
)

# Check the results
df_last[["age", "age_group"]].head(10)


,age,age_group
0,68,senior
1,42,adult
2,44,adult
3,42,adult
4,49,adult
5,32,adult
6,36,adult
7,41,adult
8,45,adult
9,31,adult


In [31]:

from scipy.stats import chi2_contingency
import pandas as pd

# Five factors from the PDF
factors = [
    "sexo",
    "age_group",
    "segmento",
    "ind_empleado",
    "tiprel_1mes"
]

results = []

for factor in factors:
    # Create contingency table
    table = pd.crosstab(
        df_last[factor],
        df_last["has_3plus"]
    )

    # Perform Chi-square test
    chi2, p, dof, expected = chi2_contingency(table)

    results.append({
        "Factor": factor,
        "Chi_square": chi2,
        "p_value": p
    })

# Display results
chi_results = pd.DataFrame(results)

chi_results


,Factor,Chi_square,p_value
0,sexo,3412.231367,0.000000e+00
1,age_group,61002.202414,0.000000e+00
2,segmento,124374.548799,0.000000e+00
3,ind_empleado,1343.689088,1.120822e-289
4,tiprel_1mes,183808.591523,0.000000e+00


In [32]:

# Sort factors by Chi-square statistic
chi_results_sorted = chi_results.sort_values(
    by="Chi_square",
    ascending=False
)

display(chi_results_sorted)

# Find the factor with the highest Chi-square
best_factor = chi_results_sorted.iloc[0]

print("Factor with highest Chi-square:", best_factor["Factor"])
print("Chi-square:", best_factor["Chi_square"])
print("p-value:", best_factor["p_value"])


,Factor,Chi_square,p_value
4,tiprel_1mes,183808.591523,0.000000e+00
2,segmento,124374.548799,0.000000e+00
1,age_group,61002.202414,0.000000e+00
0,sexo,3412.231367,0.000000e+00
3,ind_empleado,1343.689088,1.120822e-289


Factor with highest Chi-square: tiprel_1mes
Chi-square: 183808.5915229053
p-value: 0.0
